# Task 6, File 2: LDA on `text` (Text Model)

Runs after `00_build_dataset_and_config.ipynb` finishes. Fully independent of
`01b_lda_hashtags.ipynb` -- can run concurrently.

Full design reasoning: `TASK6_LDA_design_doc.md`. Exact steps: `TASK6_LDA_STEP_BY_STEP.md`.

## Step 1: Load the dataset and shared config

In [ ]:
import pandas as pd
import json
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

df_full = pd.read_csv("../../output/cleaned_data/non_trend_english.csv")
print(df_full.shape)  # expect (73933, 19)

with open("lda_config.json", "r") as f:
    config = json.load(f)

## Step 2: Build the vectorizer from the shared config

Preprocessing: lowercase, strip URLs, strip non-letter characters, POS-aware lemmatization via
NLTK's `WordNetLemmatizer` (POS tagging needed for verb forms to lemmatize correctly, e.g.
"baking"/"baked"/"bakes" -> "bake" -- a plain default-noun lemmatizer would leave these
unchanged). `ngram_range=(1,1)` (unigrams only) -- see `00`'s Step 0 note and
`TASK6_LDA_design_doc.md` for why the earlier `(1,2)` bigram-exclusion draft was dropped.

In [2]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

token_pattern = re.compile(r"\b[a-z]{2,}\b")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text

def lemmatize_tokenizer(text):
    tokens = token_pattern.findall(text)
    tagged = pos_tag(tokens)
    return [lemmatizer.lemmatize(tok, get_wordnet_pos(tag)) for tok, tag in tagged]

In [3]:
custom_stopwords = (config["tokenization_fragments"] + config["platform_artifacts"] +
                     config["recipe_boilerplate"] + config["units_measurements"] +
                     config["trend_single_words"])
raw_stopwords = list(ENGLISH_STOP_WORDS) + custom_stopwords

# Lemmatize the stopword list itself, the same way tokens get lemmatized: CountVectorizer
# filters stop_words AFTER the custom tokenizer runs (which includes lemmatization), so an
# inflected stopword ("redacted", "minutes", "further") never matches its lemmatized token
# form ("redact", "minute", "far") and leaks through. No sentence context exists for a bare
# stopword, so compute the lemma under all 4 POS categories and keep every variant -- that
# matches regardless of which POS the tokenizer assigns to it in real text.
def lemma_variants(word):
    return {lemmatizer.lemmatize(word, pos) for pos in
            (wordnet.NOUN, wordnet.VERB, wordnet.ADJ, wordnet.ADV)}

all_stopwords = set(raw_stopwords)
for w in raw_stopwords:
    all_stopwords.update(lemma_variants(w))
all_stopwords = list(all_stopwords)

vectorizer = CountVectorizer(
    preprocessor=preprocess,
    tokenizer=lemmatize_tokenizer,
    stop_words=all_stopwords,
    ngram_range=tuple(config["ngram_range"]),
    min_df=config["min_df"],
    max_df=config["max_df"],
)

corpus_texts = df_full['text']
text_dtm = vectorizer.fit_transform(corpus_texts)
feature_names_text = vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(feature_names_text)}")
print(f"Document-term matrix shape: {text_dtm.shape}")

C:\Users\minni\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vocabulary size: 31247
Document-term matrix shape: (73933, 31247)


No trimming step is needed here -- with `ngram_range=(1,1)`, `vectorizer.get_feature_names_out()`
already matches the document-term matrix's columns exactly.

## Step 3: Test K, saving incrementally, STOP for review before finalizing

Each K's model and coherence score save to disk immediately after fitting; the loop skips any K
already completed if this notebook is rerun. Coherence is real topic coherence (`gensim`'s `c_v`
measure via `CoherenceModel`), not sklearn's `.score()` (log-likelihood, a different metric that
does not measure whether a topic's top words are semantically related).

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import pickle
import os
import csv

os.makedirs("../../output/lda", exist_ok=True)
os.makedirs("../../output/lda/model_checkpoints", exist_ok=True)

coherence_log_path = "../../output/lda/model_checkpoints/coherence_log_text.csv"

def load_completed_k(log_path):
    if not os.path.exists(log_path):
        return set()
    completed = set()
    with open(log_path, "r") as f:
        for row in csv.DictReader(f):
            completed.add(int(row["k"]))
    return completed

completed_k = load_completed_k(coherence_log_path)
print(f"Already completed K values (will skip): {completed_k}")

if not os.path.exists(coherence_log_path):
    with open(coherence_log_path, "w", newline="") as f:
        csv.writer(f).writerow(["k", "coherence"])

In [5]:
# Tokenized corpus for gensim coherence -- built once via the same analyzer the vectorizer
# was fit with, so tokens exactly match the vocabulary (no separate preprocessing to drift)
analyzer = vectorizer.build_analyzer()
tokenized_text = [analyzer(doc) for doc in corpus_texts]
gensim_dictionary_text = Dictionary(tokenized_text)
print(f"Tokenized {len(tokenized_text)} documents for coherence scoring")

Tokenized 73933 documents for coherence scoring


In [ ]:
for k in config["k_values_to_test"]:
    if k in completed_k:
        print(f"K={k}: already completed, skipping")
        continue

    lda = LatentDirichletAllocation(n_components=k, random_state=42)
    lda.fit(text_dtm)

    topics = []
    for topic in lda.components_:
        top_idx = topic.argsort()[-15:][::-1]
        topics.append([feature_names_text[i] for i in top_idx])

    cm = CoherenceModel(topics=topics, texts=tokenized_text,
                         dictionary=gensim_dictionary_text, coherence="c_v", processes=1)
    coherence_score = cm.get_coherence()

    with open(f"../../output/lda/model_checkpoints/model_text_k{k}.pkl", "wb") as f:
        pickle.dump(lda, f)

    with open(coherence_log_path, "a", newline="") as f:
        csv.writer(f).writerow([k, coherence_score])

    print(f"K={k}: coherence = {coherence_score:.4f}, model saved")

**Coherence vs. K, and top 15 words per topic at each K -- STOP here for review.**

In [ ]:
results_text = {}
for k in config["k_values_to_test"]:
    with open(f"../../output/lda/model_checkpoints/model_text_k{k}.pkl", "rb") as f:
        model = pickle.load(f)
    with open(coherence_log_path, "r") as f:
        coherence = next(float(row["coherence"]) for row in csv.DictReader(f) if int(row["k"]) == k)
    results_text[k] = {"model": model, "coherence": coherence}

for k, r in results_text.items():
    print(f"K={k}: coherence = {r['coherence']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

k_values = list(results_text.keys())
coherence_scores = [results_text[k]["coherence"] for k in k_values]

plt.figure(figsize=(8, 5))
plt.plot(k_values, coherence_scores, marker="o")
plt.xlabel("Number of topics (K)")
plt.ylabel("Coherence score")
plt.title("Topic Coherence vs. K (Text Model)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../../output/lda/lda_coherence_vs_k_text.png", dpi=150)
plt.show()

In [9]:
for k in config["k_values_to_test"]:
    model = results_text[k]["model"]
    print(f"===== K={k} (coherence={results_text[k]['coherence']:.4f}) =====")
    for topic_idx, topic in enumerate(model.components_):
        top_idx = topic.argsort()[-15:][::-1]
        top_words = [feature_names_text[i] for i in top_idx]
        print(f"  Topic {topic_idx}: {top_words}")
    print()

===== K=5 (coherence=0.5839) =====
  Topic 0: ['feta', 'salad', 'olive', 'cheese', 'tomato', 'pepper', 'fresh', 'cucumber', 'red', 'avocado', 'chicken', 'lemon', 'onion', 'greek', 'dress']
  Topic 1: ['cook', 'pepper', 'garlic', 'use', 'onion', 'mix', 'bake', 'pan', 'heat', 'chop', 'powder', 'egg', 'sauce', 'water', 'cheese']
  Topic 2: ['quarantinecooking', 'quarantine', 'cook', 'follow', 'foodie', 'foodporn', 'homemade', 'home', 'quarantinelife', 'foodphotography', 'stayhome', 'foodblogger', 'share', 'foodstagram', 'lockdown']
  Topic 3: ['chocolate', 'cake', 'bake', 'quarantinebaking', 'pizza', 'cream', 'quarantinecooking', 'cheese', 'cooky', 'gram', 'dessert', 'butter', 'feedfeed', 'cookie', 'chip']
  Topic 4: ['feta', 'cheese', 'tomato', 'meal', 'eat', 'dish', 'use', 'egg', 'know', 'try', 'week', 'chicken', 'dinner', 'cook', 'breakfast']

===== K=10 (coherence=0.5834) =====
  Topic 0: ['salad', 'feta', 'olive', 'tomato', 'cheese', 'fresh', 'cucumber', 'pepper', 'dress', 'red', 'le


  Topic 4: ['feta', 'roast', 'dish', 'tomato', 'lamb', 'cheese', 'greek', 'serve', 'dip', 'hummus', 'potato', 'spinach', 'salad', 'sauce', 'dinner']
  Topic 5: ['protein', 'meal', 'healthy', 'fat', 'eat', 'calorie', 'health', 'nutrition', 'diet', 'carbs', 'low', 'lunch', 'healthyfood', 'help', 'high']
  Topic 6: ['keto', 'cheese', 'order', 'feta', 'delivery', 'pm', 'week', 'll', 'cook', 'available', 'ketodiet', 'meal', 'lowcarb', 'tag', 'follow']
  Topic 7: ['pepper', 'cook', 'garlic', 'onion', 'chop', 'heat', 'pan', 'sauce', 'olive', 'use', 'water', 'mix', 'serve', 'oven', 'tomato']
  Topic 8: ['foodie', 'instafood', 'quarantinecooking', 'foodstagram', 'foodporn', 'foodphotography', 'tomato', 'foodblogger', 'foodlover', 'yummy', 'follow', 'eeeeeats', 'yum', 'gram', 'homemade']
  Topic 9: ['bake', 'chocolate', 'cake', 'sugar', 'butter', 'flour', 'cream', 'quarantinebaking', 'cooky', 'vanilla', 'dessert', 'chip', 'use', 'cookie', 'mix']
  Topic 10: ['quarantinecooking', 'feedfeed', 'gr

**STOP.** Do not pick a final K and proceed to Step 4 without explicit project-owner
confirmation.

## Step 4: Fit (or reuse) the final model

Run only after `CONFIRMED_K` is set following project-owner review of Step 3's results above.

In [ ]:
CONFIRMED_K = 20  # confirmed by project owner

checkpoint_path = f"../../output/lda/model_checkpoints/model_text_k{CONFIRMED_K}.pkl"

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "rb") as f:
        final_lda_text = pickle.load(f)
else:
    final_lda_text = LatentDirichletAllocation(n_components=CONFIRMED_K, random_state=42)
    final_lda_text.fit(text_dtm)

with open("../../output/lda/model_checkpoints/final_lda_text.pkl", "wb") as f:
    pickle.dump(final_lda_text, f)

## Step 5: Save visualizations (top words, prevalence, pyLDAvis) and the vectorizer/DTM

In [ ]:
def plot_top_words(model, feature_names, n_top_words, title, filename):
    n_topics = model.n_components
    n_cols = 4
    n_rows = int(np.ceil(n_topics / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_idx = topic.argsort()[-n_top_words:]
        top_features = [feature_names[i] for i in top_features_idx]
        weights = topic[top_features_idx]
        ax = axes[topic_idx]
        ax.barh(top_features, weights)
        ax.set_title(f"Topic {topic_idx}")
    for j in range(n_topics, len(axes)):
        fig.delaxes(axes[j])
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()

plot_top_words(final_lda_text, feature_names_text, 15,
                "Top Words per Topic (Text Model)",
                "../../output/lda/lda_top_words_text.png")

def plot_topic_prevalence(model, dtm, title, filename):
    doc_topic = model.transform(dtm)
    dominant_topic = doc_topic.argmax(axis=1)
    counts = pd.Series(dominant_topic).value_counts().sort_index()
    plt.figure(figsize=(10, 5))
    plt.bar(counts.index.astype(str), counts.values)
    plt.xlabel("Topic")
    plt.ylabel("Number of posts where this topic is dominant")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()

plot_topic_prevalence(final_lda_text, text_dtm, "Topic Prevalence (Text Model)",
                       "../../output/lda/lda_topic_prevalence_text.png")

In [ ]:
import pyLDAvis

doc_topic_dists = final_lda_text.transform(text_dtm)
doc_topic_dists = doc_topic_dists / doc_topic_dists.sum(axis=1, keepdims=True)
topic_term_dists = final_lda_text.components_ / final_lda_text.components_.sum(axis=1, keepdims=True)
doc_lengths = np.asarray(text_dtm.sum(axis=1)).flatten()
term_frequency = np.asarray(text_dtm.sum(axis=0)).flatten()

panel_text = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=feature_names_text,
    term_frequency=term_frequency,
)
pyLDAvis.save_html(panel_text, "../../output/lda/lda_visualization_text.html")

Save the vectorizer, document-term matrix, and feature names too, not just the model --
`02_interpret_topics.ipynb` runs later as a separate process and needs these for its
example-post lookups.

In [ ]:
from scipy.sparse import save_npz

with open("../../output/lda/model_checkpoints/vectorizer_text.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
save_npz("../../output/lda/model_checkpoints/text_dtm.npz", text_dtm)
np.save("../../output/lda/model_checkpoints/feature_names_text.npy", feature_names_text)
print("Saved vectorizer, document-term matrix, and feature names.")

In [5]:
import matplotlib.pyplot as plt

In [5]:
import matplotlib.pyplot as plt